# Olist Sellers Agent

Foco exclusivo na análise dos sellers como clientes da Olist, com métricas de receita, ticket médio, frete e performance por categoria.

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore')

## Carregar dados relevantes dos sellers

base_path = '/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/'
orders = pd.read_csv(base_path + 'olist_orders_dataset.csv')
order_items = pd.read_csv(base_path + 'olist_order_items_dataset.csv')
products = pd.read_csv(base_path + 'olist_products_dataset.csv')
sellers = pd.read_csv(base_path + 'olist_sellers_dataset.csv')
payments = pd.read_csv(base_path + 'olist_order_payments_dataset.csv')
product_category = pd.read_csv(base_path + 'product_category_name_translation.csv')
sellers['seller_label'] = 'seller_' + sellers['seller_id'].astype(str)

print('Shapes:')
print('orders', orders.shape)
print('order_items', order_items.shape)
print('products', products.shape)
print('payments', payments.shape)
print('sellers', sellers.shape)

## Preparar a tabela mestre de sellers

items_products = pd.merge(order_items, products, on='product_id', how='left')
items_products = pd.merge(items_products, sellers[['seller_id', 'seller_label']], on='seller_id', how='left')
seller_master = pd.merge(items_products, orders[['order_id', 'customer_id', 'order_status']], on='order_id', how='left')
seller_master = pd.merge(seller_master, payments[['order_id', 'payment_type', 'payment_value']], on='order_id', how='left')
seller_master = pd.merge(seller_master, product_category, on='product_category_name', how='left')

seller_metrics = (
    seller_master.groupby(['seller_label'])
    .agg(
        revenue=('price', 'sum'),
        order_count=('order_id', 'nunique'),
        avg_order_value=('price', 'mean'),
        avg_freight=('freight_value', 'mean'),
        avg_payment_value=('payment_value', 'mean')
    )
    .reset_index()
)

seller_metrics.head(10)

## Top sellers por receita

top_sellers = seller_metrics.sort_values('revenue', ascending=False).head(10)
plt.figure(figsize=(12, 6))
sns.barplot(data=top_sellers, x='revenue', y='seller_label', palette='coolwarm')
plt.title('Top 10 sellers por receita total')
plt.xlabel('Receita total (R$)')
plt.ylabel('Seller')
plt.tight_layout()
plt.show()

## Métricas de eficiência de sellers

plt.figure(figsize=(12, 6))
sns.scatterplot(data=seller_metrics, x='avg_order_value', y='avg_freight', size='order_count', legend=False, alpha=0.8)
plt.title('Ticket médio x frete médio por seller')
plt.xlabel('Ticket médio (R$)')
plt.ylabel('Frete médio (R$)')
plt.tight_layout()
plt.show()

## Categorias de produto relevantes por seller

seller_category = (
    seller_master.groupby(['seller_label', 'product_category_name'])
    .agg(revenue=('price', 'sum'), order_count=('order_id', 'nunique'))
    .reset_index()
)
top_categories_by_seller = seller_category.sort_values('revenue', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_categories_by_seller, x='revenue', y='product_category_name', palette='viridis')
plt.title('Top 10 categorias por receita entre sellers')
plt.xlabel('Receita total (R$)')
plt.ylabel('Categoria')
plt.tight_layout()
plt.show()

### Conclusão do escopo de sellers

Este notebook foca em métricas que ajudam a entender os sellers como clientes da Olist e identificar oportunidades de crescimento e eficiência.